# 09: All experiment results in pictures

Run this notebook on a **CPU runtime**. It reads the tracked experiment archive and any saved full-sequence or paired L9 results on Drive. It does not train or load checkpoints. Each chart says what it shows and what it cannot establish. Missing Drive files are listed, not filled in with guesses.

The archive contains reported or transcribed observations. The Drive sections read original JSON files when available. Charts are saved as PNG files under `E08_forgetting_trace/analysis/experiment_overview` on Drive.

## SETUP-01: Load the archive and saved runs


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch

if torch.cuda.is_available():
    raise RuntimeError("Choose a CPU runtime for this reporting notebook.")

REPO = Path.cwd()
if not (REPO / "docs/experiments/registry.json").is_file():
    REPO = Path("/content/uncle-diagnostics")
    if not REPO.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", "main",
            "https://github.com/sumitasthana/CARK.git", str(REPO)
        ], check=True)
    elif not (REPO / "docs/experiments/registry.json").is_file():
        raise FileNotFoundError("Existing checkout has no experiment archive")
    else:
        if subprocess.check_output(
            ["git", "-C", str(REPO), "status", "--porcelain"], text=True
        ).strip():
            raise RuntimeError("Existing checkout has local changes; preserve them first")
        branch = subprocess.check_output(
            ["git", "-C", str(REPO), "branch", "--show-current"], text=True
        ).strip()
        if branch != "main":
            raise RuntimeError("Existing checkout is not on main")
        subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(REPO), "merge", "--ff-only", "FETCH_HEAD"], check=True)

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")

ARCHIVE = REPO / "docs/experiments"
DRIVE_ROOT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace")
if not DRIVE_ROOT.is_dir():
    DRIVE_ROOT = None
FULL_SEQUENCE_DIRS = ([DRIVE_ROOT / "full_sequence_27step_candidate"]
                      if DRIVE_ROOT else [])
PAIR_ROOT = (DRIVE_ROOT / "paired_L9_probe/same_runtime_paired"
             if DRIVE_ROOT else None)
OUTPUT = (DRIVE_ROOT / "analysis/experiment_overview" if DRIVE_ROOT else None)
if OUTPUT:
    OUTPUT.mkdir(parents=True, exist_ok=True)

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def save_plot(name):
    if OUTPUT:
        path = OUTPUT / name
        plt.savefig(path, dpi=150, bbox_inches="tight")
        print("Saved:", path)
    plt.show()

registry = read_json(ARCHIVE / "registry.json")
traces = pd.read_csv(ARCHIVE / "forgetting_traces.csv")
experiments = {item["id"]: item for item in registry["experiments"]}
print("Archive records:", len(experiments))
print("Forgetting observations:", len(traces))
print("Drive found:", DRIVE_ROOT is not None)


## ARCHIVE-01: What has been tried?

This table includes every archived record. A numerical check tests a calculation, not whether the model forgot a task. The early learning and forgetting runs are separate runs, so compare their outcomes without treating them as one continuous training path.


In [ ]:
display(pd.DataFrame([{
    "record": item["id"],
    "type": item["category"],
    "decision": item["decision"],
} for item in registry["experiments"]]))

early = [experiments[name] for name in ("E04", "E05", "E06", "E07")]
labels = [item["id"] for item in early]
x = np.arange(len(labels))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, task, title in zip(
    axes, ("task3", "task0"),
    ("Task 3: lower after U3 is desired", "Task 0: keep accuracy high"),
):
    before = [item["observations"][f"{task}_before_forgetting_pct"]
              for item in early]
    after = [item["observations"][f"{task}_after_forgetting_pct"]
             for item in early]
    ax.bar(x - 0.18, before, width=0.36, label="Before U3", color="#547aa5")
    ax.bar(x + 0.18, after, width=0.36, label="After U3", color="#dd8452")
    ax.set_xticks(x, labels)
    ax.set_title(title)
    ax.set_ylim(0, 55)
    ax.set_ylabel("Validation accuracy (%)")
    ax.grid(axis="y", alpha=0.2)
axes[0].legend()
fig.suptitle("Early forgetting runs: task 3 fell, and task 0 fell too")
fig.tight_layout()
save_plot("01_early_forgetting.png")

baseline = experiments["direct-baseline"]["observations"]["validation_accuracy_pct_by_epoch"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(range(1, len(baseline) + 1), baseline, marker="o")
axes[0].set_title("Direct ResNet50: task 3 during learning")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Validation accuracy (%)")
learned = [
    ("E02 task 3", experiments["E02"]["observations"]["task3_final_accuracy_pct"]),
    ("E03 task 3", experiments["E03"]["observations"]["task3_after_task0_learning_pct"]),
    ("E03 task 0", experiments["E03"]["observations"]["task0_final_accuracy_pct"]),
]
axes[1].bar([name for name, _ in learned], [value for _, value in learned],
            color=["#4c72b0", "#4c72b0", "#dd8452"])
axes[1].set_title("Hypernetwork: recorded learning results")
axes[1].set_ylabel("Validation accuracy (%)")
for ax in axes:
    ax.set_ylim(0, 65)
    ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_plot("01b_learning_results.png")
print("Direct training and hypernetwork runs use different setups; bars are not a controlled comparison.")


## TRACE-01: Did any forgetting step protect task 0 and lower task 3?

Each panel is one saved forgetting trace. The blue line is task 3, which we want to lower. The orange line is task 0, which we want to keep near its starting value. The shaded area marks less than five points of task 0 change. The dashed line marks the original 12% task 3 target. These traces start from the same recorded accuracies, but their settings differ.


In [ ]:
run_order = ["E08", "E09", "E10", "E11", "E12", "E13", "E14", "E15",
             "forgetting-30step-20260929"]
missing_traces = [name for name in run_order
                  if name not in set(traces["run"])]
if missing_traces:
    print("Missing traces:", missing_traces)
fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharey=True)
for ax, run in zip(axes.flat, run_order):
    data = traces[traces["run"] == run].sort_values("step")
    if data.empty:
        ax.set_title(run + " (unavailable)")
        continue
    start = data.iloc[0]["task0_accuracy_pct"]
    ax.axhspan(start - 5, start + 5, color="#dd8452", alpha=0.12)
    ax.axhline(12, color="#4c72b0", linestyle="--", linewidth=1)
    ax.plot(data["step"], data["task3_accuracy_pct"],
            color="#4c72b0", label="Task 3")
    ax.plot(data["step"], data["task0_accuracy_pct"],
            color="#dd8452", label="Task 0")
    ax.set_title(run.replace("forgetting-30step-20260929", "30-step run"))
    ax.set_xlabel("Forgetting update")
    ax.set_ylabel("Accuracy (%)")
    ax.grid(alpha=0.2)
axes[0, 0].legend(loc="upper right")
fig.suptitle("Forgetting traces: no recorded step passed both original checks")
fig.tight_layout()
save_plot("02_forgetting_traces.png")


## SEQUENCE-01: What happened to every task in the full sequence?

Each square is one task's validation accuracy after a request. A white square means that task had not been evaluated yet. Read across a row to see one task over time; read down a column to see the model after one request. The chart includes every task recorded in the saved history, not just task 0. The lines below show overall retention and spill after forget requests.


In [ ]:
full_runs = []
for folder in FULL_SEQUENCE_DIRS:
    histories = sorted(folder.glob("history_*.json")) if folder.is_dir() else []
    if len(histories) != 1:
        print("Full-sequence history unavailable:", folder)
        continue
    stem = histories[0].stem.removeprefix("history_")
    summary_path = folder / f"summary_{stem}.json"
    if not summary_path.is_file():
        print("Full-sequence summary unavailable:", summary_path)
        continue
    history = read_json(histories[0])
    numbers = read_json(summary_path)
    if not history:
        print("Empty history:", histories[0])
        continue
    full_runs.append((folder.name, history, numbers))

for run_name, history, numbers in full_runs:
    tasks = sorted({task for row in history for task in row["after"]}, key=int)
    matrix = np.array([[row["after"].get(task, np.nan) for row in history]
                       for task in tasks], dtype=float)
    fig, ax = plt.subplots(figsize=(max(11, len(history) * 0.43),
                                   max(5, len(tasks) * 0.38)))
    cmap = plt.colormaps["viridis"].copy()
    cmap.set_bad("#f1f1f1")
    image = ax.imshow(matrix, aspect="auto", vmin=0, vmax=100, cmap=cmap)
    ax.set_yticks(range(len(tasks)), [f"Task {task}" for task in tasks])
    ax.set_xticks(range(len(history)),
                  [f"{row['index']}: {'L' if row['action'] == 'learn' else 'U'}{row['task']}"
                   for row in history], rotation=90)
    ax.set_xlabel("Request in order")
    ax.set_title(f"{run_name}: accuracy of each task after each request")
    fig.colorbar(image, ax=ax, label="Validation accuracy (%)")
    fig.tight_layout()
    save_plot(f"03_{run_name}_all_tasks.png")

    retained = []
    spills = []
    for row in history:
        active = [task for task in row.get("seen", [])
                  if task not in row.get("forgotten", [])]
        retained.append(np.mean([row["after"][task] for task in active])
                        if active else np.nan)
        spills.append(sum(abs(row["after"][task] - value)
                          for task, value in row["before"].items()
                          if task != row["task"])
                      if row["action"] == "forget" else np.nan)
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(range(len(history)), retained, marker="o")
    axes[0].set_ylabel("Mean retained accuracy (%)")
    axes[1].bar(range(len(history)), spills, color="#dd8452")
    axes[1].set_ylabel("Spill at forget request (points)")
    axes[1].set_xlabel("Request number")
    for ax in axes:
        ax.grid(axis="y", alpha=0.2)
    fig.suptitle(f"{run_name}: retention and damage during forgetting")
    fig.tight_layout()
    save_plot(f"04_{run_name}_retention_and_spill.png")
    print(run_name, "saved summary:", {
        key: numbers.get(key) for key in
        ("retain_accuracy", "forget_accuracy", "mean_spill", "mean_relapse")
    })
if not full_runs:
    print("No original full-sequence JSON was found. The archive overview above remains available.")

old = experiments["full-sequence-20260921"]["observations"]
sequence_summaries = [
    {"run": "Earlier full run (reported)", "retained": old["retained_accuracy_pct"],
     "forgotten": old["forgotten_accuracy_pct"], "spill": old["mean_spill"]}
]
sequence_summaries += [
    {"run": name, "retained": numbers.get("retain_accuracy"),
     "forgotten": numbers.get("forget_accuracy"), "spill": numbers.get("mean_spill")}
    for name, _, numbers in full_runs
]
display(pd.DataFrame(sequence_summaries))
summary_table = pd.DataFrame(sequence_summaries).set_index("run")
axes = summary_table[["retained", "forgotten", "spill"]].plot.bar(
    subplots=True, layout=(1, 3), figsize=(12, 4), rot=20, legend=False,
    color=["#4c72b0", "#dd8452", "#c44e52"])
for ax, title in zip(axes.flat, ("Retained accuracy: higher", "Forgotten accuracy: near chance",
                                "Mean spill: lower")):
    ax.set_title(title)
    ax.set_ylabel("Percent or points")
    ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
save_plot("04b_full_sequence_summaries.png")


## PAIR-01: Does L9 hurt task 0 more after U3?

These charts use the saved two-arm L9 runs. They show task 0 before and after L9, and task 9 after learning. If several pairs are present, each remains a separate observation. The weight chart is shown only when a saved component diagnostic exists. Similar total weight movement does not mean the same individual weights moved.


In [ ]:
pair_rows = []
component_rows = []
if PAIR_ROOT is not None and PAIR_ROOT.is_dir():
    for folder in sorted(path for path in PAIR_ROOT.iterdir() if path.is_dir()):
        arms = {}
        for arm in ("without_u3", "after_u3"):
            histories = sorted((folder / arm).glob("history_*.json"))
            if len(histories) != 1:
                print("Incomplete pair arm:", folder.name, arm)
                break
            history = read_json(histories[0])
            if not history or (history[-1]["action"], history[-1]["task"]) != ("learn", "9"):
                print("No final L9 record:", folder.name, arm)
                break
            arms[arm] = history[-1]
        if len(arms) != 2:
            continue
        for arm, row in arms.items():
            pair_rows.append({
                "pair": folder.name, "arm": arm,
                "task0_before": row["before"].get("0"),
                "task0_after": row["after"].get("0"),
                "task9_after": row["after"].get("9"),
            })
        component_path = next((path for path in (
            folder / "component_changes.json", folder / "task0_component_changes.json")
            if path.is_file()), None)
        if component_path:
            details = read_json(component_path).get("arms", {})
            for arm, data in details.items():
                weights = (data.get("generated_weights")
                           or data.get("task0_generated_weights") or {})
                for group, values in weights.items():
                    component_rows.append({
                        "pair": folder.name, "arm": arm, "group": group,
                        "relative_change_pct": values.get("relative_l2_pct"),
                    })
else:
    print("Paired L9 folder unavailable")

pair_table = pd.DataFrame(pair_rows)
if not pair_table.empty:
    pair_table["task0_loss_points"] = (pair_table["task0_before"]
                                        - pair_table["task0_after"])
    display(pair_table)
    x = np.arange(len(pair_table))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(x - 0.18, pair_table["task0_before"], 0.36,
                label="Before L9", color="#547aa5")
    axes[0].bar(x + 0.18, pair_table["task0_after"], 0.36,
                label="After L9", color="#dd8452")
    axes[0].set_title("Task 0: how much did L9 hurt?")
    axes[0].set_ylabel("Validation accuracy (%)")
    axes[0].legend()
    axes[1].bar(x, pair_table["task9_after"], color="#55a868")
    axes[1].set_title("Task 9: what did L9 learn?")
    axes[1].set_ylabel("Validation accuracy (%)")
    for ax in axes:
        ax.set_xticks(x, [f"{r.pair[-6:]}\n{r.arm}" for r in pair_table.itertuples()],
                      rotation=45, ha="right")
        ax.set_ylim(0, 55)
        ax.grid(axis="y", alpha=0.2)
    fig.tight_layout()
    save_plot("05_paired_L9_accuracy.png")
    display(pair_table.pivot(index="pair", columns="arm",
                             values="task0_loss_points"))
else:
    print("No complete paired L9 result found")

component_table = pd.DataFrame(component_rows)
if not component_table.empty:
    display(component_table)
    for pair_id, data in component_table.groupby("pair"):
        chart = data.pivot(index="group", columns="arm",
                           values="relative_change_pct")
        ax = chart.plot.bar(figsize=(8, 4), rot=0,
                            title=f"{pair_id}: task 0 generated weight change")
        ax.set_ylabel("Relative L2 change (%)")
        ax.grid(axis="y", alpha=0.2)
        plt.tight_layout()
        save_plot(f"06_{pair_id}_component_change.png")
else:
    print("No saved component diagnostic found")


## NUMERIC-01: Calculation checks and data limits

These checks test code or formulas. They do not measure whether the full model forgets selectively. The archive is a transcription; the Drive charts read saved run JSON. A single paired result cannot establish a general U3 effect.


In [ ]:
numerical = [item for item in registry["experiments"]
             if item["category"] == "numerical check"]
display(pd.DataFrame([{
    "record": item["id"],
    "decision": item["decision"],
    "what_was_measured": ", ".join(item["observations"]),
} for item in numerical]))
print("Archived observations are reported or transcribed, not re-run here.")
print("Original Drive histories are used only when their files are present.")
if OUTPUT:
    print("Charts saved under:", OUTPUT)
if drive is not None:
    drive.flush_and_unmount(timeout_ms=600000)
    print("Drive flushed. CPU report complete.")
